<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-13 14:36:04
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.27%
Est FTT Value:   2.74 C
Future Upside:   1.20 C (78.09%)
=================== PERFORMANCE ==========================
Current PnL:             -41.58 L (-21.66%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    7.65 L (5.13%)
CY Booked PnL:           6.36 L (4.14%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               -1.39 L (-0.89%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.22 L
Total Loss:               -43.79 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-2.55,-26.54,36.26,0.09,26843.0,-26749.0,74028.0,770.00,-10.69,56.0,M-SC,1.92,250.0,-1.00,0.48,33.24,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.42,-21.56,27.47,-0.01,27539.0,-27554.0,100251.0,2282.00,787.00,54.0,M-SC,19.66,231.0,-1.00,0.65,70.64,OX40N,BTT,AC
65,SIS,-0.35,-10.99,36.24,21.26,35814.0,-12207.0,98825.0,528.00,90.04,47.0,H-SC,6.56,168.0,-0.34,0.64,46.94,OX40N,NTT,MISC
76,UNITDSPR,-0.44,4.21,15.89,20.77,38771.0,9855.0,243999.0,1644.00,-11.11,58.0,X-MC,2.15,64.0,0.25,1.59,17.29,X40N,NTT,BREWERIES
52,MASFIN,0.16,-18.56,48.89,21.26,39013.0,-18183.0,79797.0,396.02,-37.37,52.0,H-SC,14.76,167.0,-0.47,0.52,8.72,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-0.95,6.51,33.59,42.29,101454.0,18464.0,302035.0,2782.73,11.66,56.0,X-LC,2.98,29.0,0.18,1.97,24.33,X40N,ATH,AUTO
47,JIOFIN,-1.52,-19.93,69.97,36.09,224952.0,-80047.0,321498.0,387.00,-38.60,60.0,X-LC,8.91,34.0,-0.36,2.09,11.34,XY24,BTT,FINANCE
51,MARUTI,-0.91,-5.02,32.65,25.99,105345.0,-17060.0,322650.0,17120.06,-11.59,68.0,X-LC,9.97,33.0,-0.16,2.10,15.54,XR,ATH,AUTO
4,ASIANPAINT,-1.35,-8.73,36.77,24.83,119612.0,-31118.0,325299.0,3422.40,-20.45,60.0,X-LC,4.20,30.0,-0.26,2.12,19.27,X40,ATH,PAINTS
44,INFY,-1.08,-17.61,73.16,42.66,241612.0,-70603.0,330252.0,1972.00,-40.47,55.0,X-LC,7.47,3.0,-0.29,2.15,18.23,X40,NTT,IT


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES
57,RAJOOENG,1.80,-44.88,152.60,39.24,86219.0,-46000.0,56500.0,142.72,NaN,54.0,NaN,1.51,136.0,-0.53,0.37,21.37,AR,ATH,MISC
81,VOLTAS,1.57,-11.43,63.93,45.20,108572.0,-21912.0,169830.0,1856.00,22.02,63.0,X-MC,18.61,79.0,-0.20,1.11,19.66,XY25,NTT,AC
12,BOSCH-HCIL,1.42,-21.56,27.47,-0.01,27539.0,-27554.0,100251.0,2282.00,787.00,54.0,M-SC,19.66,231.0,-1.00,0.65,70.64,OX40N,BTT,AC
3,ALKYLAMINE,0.81,-30.88,178.57,92.54,124615.0,-31178.0,69785.0,4520.85,NaN,41.0,NaN,7.57,142.0,-0.25,0.45,32.52,SR,ATH,CHEMICALS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
79,VALIANTORG,-3.88,-24.35,130.22,74.15,161160.0,-39845.0,123760.0,838.00,NaN,41.0,NaN,36.39,145.0,-0.25,0.81,84.17,XR,NTT,CHEMICALS
21,DEN,-2.92,-50.86,189.13,42.07,78497.0,-42960.0,41504.0,75.00,NaN,51.0,NaN,5.93,235.0,-0.55,0.27,12.88,AR,NTT,ENTERTAINMENT
73,TMPV,-2.89,-28.03,107.86,49.60,370181.0,-133667.0,343205.0,600.00,-85.99,54.0,X-LC,16.28,4.0,-0.36,2.23,10.51,XY24,NTT,AUTO
45,IRCTC,-2.86,-44.86,177.38,52.93,264553.0,-121363.0,149145.0,1269.00,-52.49,50.0,H-MC,12.49,119.0,-0.46,0.97,4.88,XY24,BTT,TRAVEL
26,GILLETTE,-2.76,-14.17,49.93,28.69,108807.0,-35976.0,217920.0,10891.01,-38.11,50.0,X-SC,6.07,83.0,-0.33,1.42,5.29,X40,ATH,FMCG


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES
65,SIS,-0.35,-10.99,36.24,21.26,35814.0,-12207.0,98825.0,528.00,90.04,47.0,H-SC,6.56,168.0,-0.34,0.64,46.94,OX40N,NTT,MISC
12,BOSCH-HCIL,1.42,-21.56,27.47,-0.01,27539.0,-27554.0,100251.0,2282.00,787.00,54.0,M-SC,19.66,231.0,-1.00,0.65,70.64,OX40N,BTT,AC
75,TTKPRESTIG,-2.55,-26.54,36.26,0.09,26843.0,-26749.0,74028.0,770.00,-10.69,56.0,M-SC,1.92,250.0,-1.00,0.48,33.24,OX40N,NTT,DURABLES
42,INDIGOPNTS,-1.87,-31.60,46.22,0.02,55187.0,-55159.0,119400.0,1408.00,-34.71,46.0,M-SC,7.52,222.0,-1.00,0.78,36.04,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES
66,STARHEALTH,-0.03,10.46,33.32,47.26,103464.0,29392.0,310515.0,761.00,-2.57,59.0,H-SC,3.97,173.0,0.28,2.02,34.26,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES
83,WIPRO,-0.50,-10.28,115.14,93.01,258493.0,-25737.0,224503.0,420.00,-24.11,62.0,M-LC,8.02,98.0,-0.10,1.46,24.51,XR,NTT,IT
69,TANLA,0.07,-17.36,160.38,115.17,358845.0,-47014.0,223747.0,1903.93,NaN,83.0,NaN,20.11,147.0,-0.13,1.46,97.79,AR,ATH,IT
41,INDIAMART,-1.59,-30.86,190.59,100.92,162529.0,-38059.0,85277.0,4675.60,NaN,53.0,NaN,16.53,140.0,-0.23,0.55,8.19,AR,ATH,MISC
3,ALKYLAMINE,0.81,-30.88,178.57,92.54,124615.0,-31178.0,69785.0,4520.85,NaN,41.0,NaN,7.57,142.0,-0.25,0.45,32.52,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,-1.21,-17.93,61.88,32.85,136294.0,-48124.0,220255.0,50936.07,-39.04,30.0,X-MC,3.65,55.0,-0.35,1.43,5.18,X40,ATH,APPARELS
6,AWL,-2.40,-43.47,195.44,67.02,393663.0,-154885.0,201424.0,485.00,-58.24,36.0,X-SC,9.33,81.0,-0.39,1.31,0.19,XY24,NTT,FMCG
37,HONAUT,-0.93,-21.10,91.23,50.88,138806.0,-40688.0,152150.0,58192.80,-38.37,41.0,X-SC,8.09,90.0,-0.29,0.99,15.92,X40N,ATH,ELECTRICAL
55,PGHH,-2.18,-47.29,148.57,31.02,167207.0,-100976.0,112544.0,17484.40,-61.22,49.0,X-SC,23.00,57.0,-0.60,0.73,5.95,X40,ATH,FMCG
53,NESTLEIND,0.55,6.38,21.47,29.22,71316.0,19926.0,332164.0,1540.00,-19.66,50.0,X-LC,9.24,20.0,0.28,2.16,9.18,XY25,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-0.47,-11.19,29.90,15.37,47496.0,-20005.0,158850.0,34390.26,-27.79,53.0,X-MC,0.72,58.0,-0.42,1.03,7.60,X40,ATH,PHARMA
17,CIPLA,-0.61,5.21,17.75,23.89,56149.0,15677.0,316332.0,1648.14,19.26,58.0,X-LC,0.88,18.0,0.28,2.06,18.49,X40N,ATH,PHARMA
76,UNITDSPR,-0.44,4.21,15.89,20.77,38771.0,9855.0,243999.0,1644.00,-11.11,58.0,X-MC,2.15,64.0,0.25,1.59,17.29,X40N,NTT,BREWERIES
11,BERGEPAINT,-0.82,-12.37,43.01,25.32,97143.0,-31887.0,225862.0,680.00,-23.78,51.0,X-MC,2.42,68.0,-0.33,1.47,18.44,XY24,NTT,PAINTS
38,HYUNDAI,-0.95,6.51,33.59,42.29,101454.0,18464.0,302035.0,2782.73,11.66,56.0,X-LC,2.98,29.0,0.18,1.97,24.33,X40N,ATH,AUTO


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-2.40,-43.47,195.44,67.02,393663.0,-154885.0,201424.0,485.00,-58.24,36.0,X-SC,9.33,81.0,-0.39,1.31,0.19,XY24,NTT,FMCG
54,PAGEIND,-1.21,-17.93,61.88,32.85,136294.0,-48124.0,220255.0,50936.07,-39.04,30.0,X-MC,3.65,55.0,-0.35,1.43,5.18,X40,ATH,APPARELS
26,GILLETTE,-2.76,-14.17,49.93,28.69,108807.0,-35976.0,217920.0,10891.01,-38.11,50.0,X-SC,6.07,83.0,-0.33,1.42,5.29,X40,ATH,FMCG
55,PGHH,-2.18,-47.29,148.57,31.02,167207.0,-100976.0,112544.0,17484.40,-61.22,49.0,X-SC,23.00,57.0,-0.60,0.73,5.95,X40,ATH,FMCG
28,GODREJCP,-0.68,-20.81,66.82,32.10,132039.0,-51933.0,197604.0,1452.16,-22.23,56.0,X-MC,15.70,32.0,-0.39,1.29,7.06,X40N,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PGHH,-2.18,-47.29,148.57,31.02,167207.0,-100976.0,112544.0,17484.40,-61.22,49.0,X-SC,23.00,57.0,-0.60,0.73,5.95,X40,ATH,FMCG
2,ACC,-1.74,-45.44,220.43,74.81,322445.0,-121848.0,146280.0,3906.00,-45.85,56.0,X-SC,13.33,85.0,-0.38,0.95,8.37,XY24,BTT,CEMENT
37,HONAUT,-0.93,-21.10,91.23,50.88,138806.0,-40688.0,152150.0,58192.80,-38.37,41.0,X-SC,8.09,90.0,-0.29,0.99,15.92,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-0.47,-11.19,29.90,15.37,47496.0,-20005.0,158850.0,34390.26,-27.79,53.0,X-MC,0.72,58.0,-0.42,1.03,7.60,X40,ATH,PHARMA
19,COLPAL,-0.97,-34.95,111.77,37.76,191484.0,-92045.0,171320.0,3628.04,-21.15,53.0,X-MC,14.63,62.0,-0.48,1.11,8.95,XY25,ATH,FMCG


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,0.09,-19.05,58.99,28.70,208965.0,-83357.0,354238.0,452.00,-27.09,66.0,X-LC,6.98,1.0,-0.40,2.31,11.49,X40,NTT,FMCG
71,TCS,-1.37,-27.45,74.70,26.75,264519.0,-133967.0,354108.0,4208.34,-41.29,51.0,X-LC,8.02,2.0,-0.51,2.30,22.82,X40,ATH,IT
44,INFY,-1.08,-17.61,73.16,42.66,241612.0,-70603.0,330252.0,1972.00,-40.47,55.0,X-LC,7.47,3.0,-0.29,2.15,18.23,X40,NTT,IT
73,TMPV,-2.89,-28.03,107.86,49.60,370181.0,-133667.0,343205.0,600.00,-85.99,54.0,X-LC,16.28,4.0,-0.36,2.23,10.51,XY24,NTT,AUTO
33,HCLTECH,-0.62,-6.10,33.87,25.70,114805.0,-22015.0,338958.0,1837.07,-10.50,51.0,X-LC,9.36,8.0,-0.19,2.21,35.24,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,1.42,-21.56,27.47,-0.01,27539.0,-27554.0,100251.0,2282.00,787.00,54.0,M-SC,19.66,231.0,-1.00,0.65,70.64,OX40N,BTT,AC
79,VALIANTORG,-3.88,-24.35,130.22,74.15,161160.0,-39845.0,123760.0,838.00,NaN,41.0,NaN,36.39,145.0,-0.25,0.81,84.17,XR,NTT,CHEMICALS
27,GLAXO,0.55,7.19,26.59,35.69,56798.0,14324.0,213608.0,3380.16,-19.73,56.0,X-MC,7.74,56.0,0.25,1.39,26.95,X40,ATH,PHARMA
69,TANLA,0.07,-17.36,160.38,115.17,358845.0,-47014.0,223747.0,1903.93,NaN,83.0,NaN,20.11,147.0,-0.13,1.46,97.79,AR,ATH,IT
25,FINCABLES,1.94,61.74,27.64,106.44,66004.0,91155.0,238799.0,1629.94,17.82,54.0,M-SC,23.63,220.0,1.38,1.55,80.94,OX40N,ATH,CABLES


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,0.07,-17.36,160.38,115.17,358845.0,-47014.0,223747.0,1903.93,NaN,83.0,NaN,20.11,147.0,-0.13,1.46,97.79,AR,ATH,IT
50,M&M,-1.67,4.35,12.58,17.48,39523.0,13101.0,314176.0,3762.88,-8.98,69.0,X-LC,4.93,25.0,0.33,2.04,21.49,X40N,ATH,AUTO
61,REPCOHOME,-0.92,-17.82,130.97,89.82,308379.0,-51047.0,235458.0,880.00,NaN,71.0,NaN,10.82,161.0,-0.17,1.53,27.11,XY24,NTT,FINANCE
83,WIPRO,-0.50,-10.28,115.14,93.01,258493.0,-25737.0,224503.0,420.00,-24.11,62.0,M-LC,8.02,98.0,-0.10,1.46,24.51,XR,NTT,IT
51,MARUTI,-0.91,-5.02,32.65,25.99,105345.0,-17060.0,322650.0,17120.06,-11.59,68.0,X-LC,9.97,33.0,-0.16,2.10,15.54,XR,ATH,AUTO


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.59
1,25,51.34
2,50,82.23


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.90
SC    20.45
MC    19.64
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.02
XY24     22.92
X40N     19.18
XR       10.06
XY25      9.89
OX40N     6.80
AR        3.38
SR        0.75
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    41.44
X-MC    17.25
H-SC     8.30
X-SC     6.97
M-SC     5.18
M-LC     1.46
M-MC     1.08
H-MC     0.97
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.23,-25.70,71.04
IT,12.80,-30.47,98.45
FINANCE,10.93,-21.29,72.91
AUTO,9.22,-12.77,54.19
MISC,5.63,-42.47,99.73
PAINTS,5.55,-24.28,46.60
INSURANCE,5.48,1.87,35.01
BANKS,5.26,-10.98,66.64
PHARMA,4.48,1.45,23.29


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3971831.0,19
X40,2181918.0,16
XR,1559627.0,13
X40N,1485693.0,12
AR,989525.0,6
XY25,770148.0,6
OX40N,744384.0,10
SR,297597.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2967500.0,19
X-MC,1585553.0,13
X-SC,1202884.0,6
H-SC,874865.0,8
M-SC,366055.0,6
H-MC,264553.0,1
M-LC,258493.0,1
M-MC,79951.0,1
L-MC,50280.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1299489.0      8
X-SC       XY24       716108.0      2
X-LC       XY24       595133.0      2
           X40N       577392.0      5
H-SC       XY24       532230.0      3
X-MC       X40N       531498.0      4
           X40        485648.0      5
X-LC       XY25       390141.0      3
X-MC       XY25       300056.0      2
X-SC       X40        276014.0      2
X-MC       XY24       268351.0      2
H-MC       XY24       264553.0      1
M-LC       XR         258493.0      1
H-SC       OX40N      242826.0      3
X-SC       X40N       210762.0      2
M-SC       OX40N      175573.0      4
           XY24       130092.0      1
X-LC       XR         105345.0      1
H-SC       XR          99809.0      2
M-MC       XY25        79951.0      1
M-SC       XR          60390.0      1
L-MC       XR          50280.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 35.0 seconds
